In [1]:
import numpy as np
import torchvision as thv

from torch import torch
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler, SequentialSampler

from utils import Utils
from consts import Consts
from feed_fwd_nn_model import FeedFwdNNModel
from feed_fwd_nn_params import FeedFwdNNParams
from nn_training_params import NNTrainingParams

In [2]:
train_ds = thv.datasets.MNIST(
    root="./data"
    , train=True
    , download=True
    , transform=thv.transforms.Compose(
        [
            thv.transforms.ToTensor()
            , thv.transforms.Normalize(mean=Consts.DS_MEAN, std=Consts.DS_STD)
        ]
    )
)

val_ds = thv.datasets.MNIST(
    root="./data"
    , train=False
    , download=True
    , transform=thv.transforms.Compose(
        [
            thv.transforms.ToTensor()
            , thv.transforms.Normalize(mean=Consts.DS_MEAN, std=Consts.DS_STD)
        ]
    )
)

print(len(val_ds.data))
print(len(train_ds.data))

10000
60000


In [3]:
model = FeedFwdNNModel(feed_fwd_nn_params=FeedFwdNNParams(input_dim=28*28, output_dim=10, dropout_prob=0.1))

model.train(params=NNTrainingParams(n_epochs=2, optimizer_alg="sgd", train_batch_size=len(train_ds), learning_rate=0.1, train_ds=train_ds, val_batch_size=len(val_ds), val_ds=val_ds))

val_error: 0.3315: 100%|██████████| 2/2 [00:10<00:00,  5.16s/it]


array([IterationDataPoint(iter_idx=0, epoch_idx=0, batch_idx=0, train_loss=2.557213306427002, train_error=0.9266, val_loss=1.5502381324768066, val_error=0.4696),
       IterationDataPoint(iter_idx=1, epoch_idx=1, batch_idx=0, train_loss=1.5638394355773926, train_error=0.4671333333333333, val_loss=1.1631842851638794, val_error=0.3315)],
      dtype=object)

In [ ]:
lrs = [0.1, 0.01]
dropout_ps = [0.1, 0.2, 0.5]
optimizer_algs = ["sgd", "adam"]
hidden_sizes = [[], [500], [1000], [1000, 500]]

models = {
    str(model): model for model in [
        FeedFwdNNModel(
            lr=lr
            , dropout_p=dropout_p
            , loader_val=loader_val
            , hidden_sizes=hidden_size
            , loader_train=loader_train
            , optimizer_alg=optimizer_alg
            , device_name=Utils.get_device_name()
        ) for lr in lrs for optimizer_alg in optimizer_algs for dropout_p in dropout_ps for hidden_size in hidden_sizes
    ]
}

idps = { model_name: model.train_and_validate() for model_name, model in models.items() }

In [ ]:
top_model_names = [
    kvp[0] for kvp in sorted(
        list(idps.items())
        , key=lambda kvp: min(
            kvp[1]
            , key=lambda idp: idp.validation_error
        ).validation_error
    )[:Consts.K_TOP_MODELS]
]

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=5
    , fig_size=(25, 20)
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Losses"
    , x=[idp.iter_idx for idp in idps[top_model_names[0]]]
    , yss_legend=[[f"{loss_type} of {model_name}" for loss_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.training_loss for model_idp in idps[model_name]], [model_idp.validation_loss for model_idp in idps[model_name]]] for model_name in top_model_names]
)

In [ ]:
Utils.multi_line_plot(
    x_ticks_inc=5
    , fig_size=(25, 20)
    , y_axis_label="Error"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Errors"
    , x=[idp.iter_idx for idp in idps[top_model_names[0]]]
    , yss_legend=[[f"{error_type} of {model_name}" for error_type in ["training", "validation"]] for model_name in top_model_names]
    , yss=[[[model_idp.training_error for model_idp in idps[model_name]], [model_idp.validation_error for model_idp in idps[model_name]]] for model_name in top_model_names]
)

In [ ]:
print(f"best is {top_model_names[0]} and achieves validation error of {min(idps[top_model_names[0]], key=lambda idp: idp.validation_error).validation_error}")